<a href="https://colab.research.google.com/github/alejoherrera/stellar_repo/blob/main/sdk/py/examples/colab_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Monitor as a Service — Colab Demo

**Powered by [Mivisor.com](https://mivisor.com)**

Lee anclas de obra publica desde **Stellar testnet**, verifica integridad contra **IPFS** publico, y genera un **dashboard interactivo** — todo desde Google Colab, sin servidor propio, sin API keys.

**Schema CC0** | **SDK MIT** | **0 backend dependencies**

---

Coautores: Juan Alejandro Herrera Lopez | Andres Herrera Monge (CEO Mivisor.com) | Claude (Anthropic AI assistant)


## 1. Instalar el SDK

Solo instalamos `monitor-as-a-service` — Colab ya trae `pandas`, `plotly` y `ipywidgets` pre-instalados con versiones compatibles. **NO** uses `--force-reinstall` ni instales `pandas`/`plotly` aqui: rompe el ambiente de Colab.


In [1]:
!pip install --quiet --upgrade monitor-as-a-service


**Si Colab ya tenia una version vieja de `monitor-as-a-service` cacheada** (caso comun cuando trabajas el notebook por varios dias), reinicia el runtime una sola vez:

1. Menu: `Runtime` → `Restart session` (o `Ctrl+M .`).
2. **No** hace falta volver a correr esta celda 1 — saltea directo a la celda 2.

Si despues de reiniciar igual ves `ModuleNotFoundError`, corré esta celda diagnostica:

```python
import monitor_as_a_service
print('Version:', monitor_as_a_service.__version__)  # debe decir 1.1.0 o superior
from monitor_as_a_service import Client
print('Client.available_dates exists:', hasattr(Client, 'available_dates'))
```


## 2. Conectar a Stellar testnet

El publisher publico de demostracion es la cuenta `GDRWQERI...3JCPFR`.
Cualquier dev puede consultarla sin permiso. Aqui pedimos su metadata de proyecto.


In [2]:
from monitor_as_a_service import Client

ACCOUNT = "GDRWQERI6PI3WICTGPJBFBEFRV7ZRLCWG3IRA2YZQA5ZINHPY23JCPFR"
client = Client.testnet(ACCOUNT)

project = client.project()
print(f"Proyecto on-chain: {project.name}")
print(f"Codigo: {project.code}")
print(f"Sistema productor: {project.system}")
print(f"Contraparte: {project.partner}")
print(f"URL publica: https://{project.url}")


Proyecto on-chain: Monitoreo ciudadano Circunvalacion San Jose
Codigo: circunvalacion-cr
Sistema productor: dIAra
Contraparte: LanammeUCR
URL publica: https://obrapublica.info/ciudadania


## 3. Listar todos los outputs anclados

Cada output es una observacion de la obra (foto + JSON con metadata generada por IA), anclada en una transaccion Stellar con 8 operaciones `manageData` (hash + 5 metadatos legibles + 2 CIDs IPFS).


In [3]:
import pandas as pd

outputs = list(client.outputs())
print(f"Total outputs anclados: {len(outputs)}")
print(f"Rango: {outputs[-1].output_id}  ->  {outputs[0].output_id}")
print()

df = pd.DataFrame([{
    "output_id": o.output_id,
    "datetime": o.datetime,
    "workers": o.workers,
    "machinery": o.machinery,
    "phase": o.phase,
    "image_url": o.image_url,
} for o in outputs])
df.head(10)


Total outputs anclados: 106
Rango: 20251003-073611  ->  20251129-175905



,output_id,datetime,workers,machinery,phase,image_url
0,20251129-175905,29/11/2025 17:59:05,0,ninguna,,https://gateway.pinata.cloud/ipfs/bafkreiaitem...
1,20251129-152245,29/11/2025 15:22:45,1,"Perforadora de pilotes, Grua de celesia",Cimentacion y pilotes,https://gateway.pinata.cloud/ipfs/bafkreibhybl...
2,20251129-124015,29/11/2025 12:40:15,0,"Perforadora de pilotes, Grua de celesia, Gener...",Cimentacion y pilotes,https://gateway.pinata.cloud/ipfs/bafkreidoojz...
3,20251129-095756,29/11/2025 09:57:56,0,Perforadora de pilotes,Cimentacion y pilotes,https://gateway.pinata.cloud/ipfs/bafkreign3bd...
4,20251029-175920,29/10/2025 17:59:20,0,ninguna,,https://gateway.pinata.cloud/ipfs/bafkreif6y6v...
5,20251029-060211,29/10/2025 06:02:11,2,Retroexcavadora,Estructura de concreto,https://gateway.pinata.cloud/ipfs/bafkreia5ikn...
6,20251019-155645,19/10/2025 15:56:45,1,Excavadora,Estructura de concreto,https://gateway.pinata.cloud/ipfs/bafkreihc5ld...
7,20251019-140350,19/10/2025 14:03:50,0,Compactador manual,Estructura de concreto,https://gateway.pinata.cloud/ipfs/bafkreiep4d3...
8,20251019-120515,19/10/2025 12:05:15,0,ninguna,Armado de acero,https://gateway.pinata.cloud/ipfs/bafkreihci27...
9,20251019-115949,19/10/2025 11:59:49,0,Excavadora,Estructura de concreto,https://gateway.pinata.cloud/ipfs/bafkreig6ag6...


## 4. Ver una imagen anclada

Las imagenes viven en IPFS — content-addressable, recuperables desde cualquier gateway publico. Cualquier modificacion en la imagen cambiaria el CID y rompiendo la verificacion.


In [4]:
from IPython.display import Image, display

first = outputs[0]
print(f"Output: {first.output_id}")
print(f"Fecha: {first.datetime}")
print(f"Phase: {first.phase}")
print(f"Image CID: {first.image_cid}")
print(f"Hash on-chain (img): {first.image_hash_onchain}")
print()
display(Image(url=first.image_url, width=600))


Output: 20251129-175905
Fecha: 29/11/2025 17:59:05
Phase:  
Image CID: bafkreiaitemaroudoglwh7t3eviaqmfkpub3xrb62lifalbq5n6npt724u
Hash on-chain (img): 08991808ba83719763fe7b25500830aa7d03bbc43ed2d0502c30eb7cd7cffae5



## 5. Verificar integridad criptografica

El SDK fetch-ea el JSON desde IPFS, calcula su SHA-256 canonico, y compara con el hash anclado en Stellar. Si todo cierra, el output es **provadamente autentico** — ni siquiera el publisher puede haberlo alterado retroactivamente.


In [5]:
result = client.verify(outputs[0].output_id)
print(f"Output: {result.output_id}")
print(f"JSON match: {result.json_ok}")
print(f"Image match: {result.image_ok}")
print(f"Hash JSON calculado: {result.computed_json_hash}")
print(f"Hash JSON on-chain:  {outputs[0].json_hash_onchain}")


Output: 20251129-175905
JSON match: True
Image match: True
Hash JSON calculado: 6fa8b3329feb5c68b2360b1199c39256a164bc3368cfdc5dbebc6f25fdd5cec7
Hash JSON on-chain:  6fa8b3329feb5c68b2360b1199c39256a164bc3368cfdc5dbebc6f25fdd5cec7


Verifiquemos los primeros 5 en lote:


In [6]:
for o in outputs[:5]:
    r = client.verify(o.output_id, verify_image=False)
    status = "OK" if r.json_ok else "FAIL"
    print(f"  [{status}]  {o.output_id}  -  {(o.phase or '-')[:30]}")


  [OK]  20251129-175905  -   
  [OK]  20251129-152245  -  Cimentacion y pilotes
  [OK]  20251129-124015  -  Cimentacion y pilotes
  [OK]  20251129-095756  -  Cimentacion y pilotes
  [OK]  20251029-175920  -   


## 6. Dashboard interactivo (plotly)

Construimos un dashboard inline con los datos leidos. Notese: **toda la data viene de Stellar testnet**, no de un servidor nuestro.


In [7]:
from collections import Counter
import plotly.graph_objects as go
from plotly.subplots import make_subplots

phases = Counter(o.phase or "(sin etapa)" for o in outputs)
machinery = Counter(
    m.strip() for o in outputs
    for m in (o.machinery or "").split(",")
    if m.strip() and m.strip().lower() != "ninguna"
)

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        "Outputs por etapa constructiva",
        "Maquinaria detectada (top 10)",
        "Personas trabajadoras a lo largo del tiempo",
        "Distribucion de outputs por dia",
    ),
    specs=[[{"type": "pie"}, {"type": "bar"}],
           [{"type": "scatter"}, {"type": "bar"}]],
    vertical_spacing=0.16,
)

fig.add_trace(go.Pie(labels=list(phases), values=list(phases.values()), hole=0.45, textinfo="label+percent"), row=1, col=1)

top_m = machinery.most_common(10)
fig.add_trace(go.Bar(x=[c for _, c in top_m], y=[m for m, _ in top_m], orientation="h", marker_color="#1a73e8"), row=1, col=2)

chrono = sorted(outputs, key=lambda x: x.output_id)
fig.add_trace(go.Scatter(x=[o.datetime or o.output_id for o in chrono], y=[o.workers or 0 for o in chrono], mode="lines+markers", line=dict(color="#fbbf24", width=2)), row=2, col=1)

by_day = Counter(o.output_id[:8] for o in outputs)
days = sorted(by_day)
fig.add_trace(go.Bar(x=days, y=[by_day[d] for d in days], marker_color="#10b981"), row=2, col=2)

fig.update_layout(
    title=dict(
        text=f"<b>Monitor as a Service</b> | {project.name}<br>"
             f"<span style='font-size:11px; color:#6b7280;'>Powered by Mivisor.com | Schema CC0 | Stellar testnet</span>",
        x=0.02, y=0.97, yanchor="top", font=dict(size=20),
    ),
    height=900, showlegend=False,
    margin=dict(t=145, b=40, l=40, r=40),
    paper_bgcolor="#f8fafc",
)
fig.update_annotations(yshift=-15)
fig.show()


## 7. Bonus: detectar anomalias

Ahora que tenemos los outputs como DataFrame, encontrar patrones es trivial.
Por ejemplo: dias con cero personas trabajadoras durante hora laboral (probable suspension de obra).


In [8]:
df["hour"] = df["output_id"].str.split("-").str[1].str[:2].astype(int)
labor_hours = df[(df["hour"] >= 7) & (df["hour"] <= 16)]
no_workers = labor_hours[labor_hours["workers"] == 0]
print(f"Outputs en horario laboral con 0 personas: {len(no_workers)} de {len(labor_hours)}")
no_workers[["output_id", "datetime", "phase", "machinery"]].head(10)


Outputs en horario laboral con 0 personas: 46 de 103


,output_id,datetime,phase,machinery
2,20251129-124015,29/11/2025 12:40:15,Cimentacion y pilotes,"Perforadora de pilotes, Grua de celesia, Gener..."
3,20251129-095756,29/11/2025 09:57:56,Cimentacion y pilotes,Perforadora de pilotes
7,20251019-140350,19/10/2025 14:03:50,Estructura de concreto,Compactador manual
8,20251019-120515,19/10/2025 12:05:15,Armado de acero,ninguna
9,20251019-115949,19/10/2025 11:59:49,Estructura de concreto,Excavadora
10,20251019-093414,19/10/2025 09:34:14,Estructura de concreto,Excavadora
11,20251019-070345,19/10/2025 07:03:45,Estructura de concreto,Cargador frontal
12,20251018-155615,18/10/2025 15:56:15,Estructura de concreto,Excavadora
13,20251018-120128,18/10/2025 12:01:28,Armado de acero,Retroexcavadora
15,20251018-093033,18/10/2025 09:30:33,Estructura de concreto,Generador electrico


## 8. Widgets interactivos (NUEVO en 1.1.0)

El SDK incluye widgets de ipywidgets para explorar los datos sin escribir codigo.

### 8.1 Buscar imagen anclada por fecha y hora

Calendario + sliders de hora y minuto. El SDK encuentra el output mas cercano y muestra la imagen pineada en IPFS con su metadata.


In [9]:
from monitor_as_a_service.widgets import image_at_datetime
image_at_datetime(client)


### 8.2 Generar dashboard por rango de fechas

Dos DatePickers + boton. Filtra los outputs en el rango y renderiza un dashboard plotly inline con KPIs y 4 charts.


In [10]:
from monitor_as_a_service.widgets import dashboard_for_range
dashboard_for_range(client)


### 8.3 Helpers programaticos (sin widget)

Si preferis llamar las funciones directo:


In [11]:
# Listar fechas disponibles on-chain
fechas = client.available_dates()
print(f"{len(fechas)} dias con outputs: {fechas[0]} -> {fechas[-1]}")

# Outputs de un dia especifico
octubre_3 = client.outputs_on_date("2025-10-03")
print(f"3 oct 2025: {len(octubre_3)} outputs")

# Outputs de un rango
primera_semana = client.outputs_in_range("2025-10-03", "2025-10-09")
print(f"Primera semana: {len(primera_semana)} outputs")

# Output mas cercano a un datetime
cercano = client.find_nearest("2025-10-29 14:30:00")
print(f"Mas cercano a 14:30 del 29-oct: {cercano.output_id}")


19 dias con outputs: 2025-10-03 -> 2025-11-29
3 oct 2025: 6 outputs
Primera semana: 41 outputs
Mas cercano a 14:30 del 29-oct: 20251029-175920


## 8. Que mas se puede construir

- **Bot Telegram/Discord** que postee cuando entra un output con anomalia
- **Email digest** semanal del estado de la obra
- **Mapa interactivo** con coordenadas (cuando se agreguen al schema)
- **Comparativa entre obras** monitoreadas por distintos publishers
- **Periodismo de datos** con CSV exports

El schema es CC0. Los SDKs son MIT. Forkea, copia, redistribui — el objetivo es que esto funcione sin nosotros.

**Recursos:**
- Live dashboard: https://www.obrapublica.info/stellar/dashboard
- Developer hub: https://www.obrapublica.info/stellar/dev
- Schema (CC0): https://github.com/alejoherrera/stellar_repo/blob/main/docs/SCHEMA.md
- Cuenta Stellar testnet: [`GDRWQERI...3JCPFR`](https://stellar.expert/explorer/testnet/account/GDRWQERI6PI3WICTGPJBFBEFRV7ZRLCWG3IRA2YZQA5ZINHPY23JCPFR)

---

**Powered by [Mivisor.com](https://mivisor.com)**

Coautores: Juan Alejandro Herrera Lopez ・ Andres Herrera Monge (CEO Mivisor.com) ・ Claude (Anthropic AI assistant)
